In [1]:
import boto3

s3 = boto3.client('s3')
print(s3.list_buckets())

{'ResponseMetadata': {'RequestId': 'Q1R65WBMCJYZMDYR', 'HostId': 'hXlEyh8xgol8wnFZ7j9l1S6uMYLraSdDBo2A3mWz1sQV6xEyAsz4BtnTn47v9NZUQ0MFohsys64=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'hXlEyh8xgol8wnFZ7j9l1S6uMYLraSdDBo2A3mWz1sQV6xEyAsz4BtnTn47v9NZUQ0MFohsys64=', 'x-amz-request-id': 'Q1R65WBMCJYZMDYR', 'date': 'Thu, 16 Apr 2026 01:29:39 GMT', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'Buckets': [{'Name': 'bedrock-finetune-data-sft', 'CreationDate': datetime.datetime(2026, 4, 7, 1, 12, 16, tzinfo=tzlocal()), 'BucketArn': 'arn:aws:s3:::bedrock-finetune-data-sft'}, {'Name': 'checkbuket', 'CreationDate': datetime.datetime(2026, 4, 7, 1, 8, 29, tzinfo=tzlocal()), 'BucketArn': 'arn:aws:s3:::checkbuket'}, {'Name': 'rag-demo-docs-sri', 'CreationDate': datetime.datetime(2026, 4, 8, 16, 53, 50, tzinfo=tzlocal()), 'BucketArn': 'arn:aws:s3:::rag-demo-docs-sri'}, {'Name': 'sagemaker-ap-south-1-502761807248', 'Creatio

# Read File from S3 successful

In [2]:
import pandas as pd
df = pd.read_csv('s3://rag-demo-docs-sri/test_user_credentials.csv')
df.head()
'''
Read successfull it is as expected
'''

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,Unnamed: 0,User name,Password,Console sign-in URL
0,0,test_user,test@123,https://502761807248.signin.aws.amazon.com/con...


# Upload file to S3

In [3]:
df.to_csv('s3://rag-demo-docs-sri/check_credentials.csv')

'''
Upload file to S3 denied as per policy set... it is as expected
'''

AccessDenied: An error occurred (AccessDenied) when calling the PutObject operation: User: arn:aws:sts::502761807248:assumed-role/SageMakerExecutionRole-S3Controlled/SageMaker is not authorized to perform: s3:PutObject on resource: "arn:aws:s3:::rag-demo-docs-sri/check_credentials.csv" with an explicit deny in an identity-based policy

# Delete S3 File

In [4]:
import boto3
s3 = boto3.client('s3')
s3.delete_object(Bucket='rag-demo-docs-sri', Key='test_user_credentials.csv')

'''
Delete file from S3 denied as per policy set... it is as expected
'''

AccessDenied: An error occurred (AccessDenied) when calling the DeleteObject operation: User: arn:aws:sts::502761807248:assumed-role/SageMakerExecutionRole-S3Controlled/SageMaker is not authorized to perform: s3:DeleteObject on resource: "arn:aws:s3:::rag-demo-docs-sri/test_user_credentials.csv" with an explicit deny in an identity-based policy

# Copy Data to Another Bucket

In [5]:
s3.copy_object(
    Bucket='checkbuket',
    CopySource='rag-demo-docs-sri/test_user_credentials.csv',
    Key='copied.csv'
)

'''
Copy file from one S3 bucket to other bucket also denied as per policy set... it is as expected
'''

AccessDenied: An error occurred (AccessDenied) when calling the CopyObject operation: User: arn:aws:sts::502761807248:assumed-role/SageMakerExecutionRole-S3Controlled/SageMaker is not authorized to perform: s3:PutObject on resource: "arn:aws:s3:::checkbuket/copied.csv" with an explicit deny in an identity-based policy

# Control at Multiple levels a below:
## **IAM-level access: Restricted
## **S3 data modification: Blocked
## **Cross-bucket movement: Prevented
## **Read-only analytical access: Enabled via SageMaker

In [7]:
#uncomment below line and run if there is any library error issue
! pip install boto3 python-docx PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [PyPDF2]2m1/2 [PyPDF2]


In [3]:
#Load Your Extracted Documents from your S3 bucket

In [1]:
import boto3
from io import BytesIO
from docx import Document
import PyPDF2

s3 = boto3.client('s3')
BUCKET_NAME = "rag-demo-docs-sri"  #replace with your s3 bucket name


def read_docx_from_s3(key):
    response = s3.get_object(Bucket=BUCKET_NAME, Key=key)
    file_stream = BytesIO(response['Body'].read())
    doc = Document(file_stream)
    return "\n".join([para.text for para in doc.paragraphs])


def read_pdf_from_s3(key):
    response = s3.get_object(Bucket=BUCKET_NAME, Key=key)
    file_stream = BytesIO(response['Body'].read())
    reader = PyPDF2.PdfReader(file_stream)
    
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    
    return text


def extract_text(file_key):
    if file_key.endswith(".docx"):
        return read_docx_from_s3(file_key)
    elif file_key.endswith(".pdf"):
        return read_pdf_from_s3(file_key)
    return ""


# Load all documents
response = s3.list_objects_v2(Bucket=BUCKET_NAME)
files = [obj['Key'] for obj in response.get('Contents', [])]

documents = {}
for file in files:
    documents[file] = extract_text(file)

print(list(documents.keys()))

['HR-POL-008_Leave_Policy.docx', 'HR-POL-009_Remote_Work_Policy.docx', 'test_user_credentials.csv']


In [2]:
#Basic Chunking (Start Simple)

In [3]:
def basic_chunking(text, chunk_size=500):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

In [4]:
'''
Here we are blindly splitting text. ...
This often breaks context and reduces answer quality.
'''

sample_text = list(documents.values())[0]

chunks = basic_chunking(sample_text)

print("Total chunks:", len(chunks))
print(chunks[0])

Total chunks: 8

ABC Solutions Pvt. Ltd.
ABC Solutions Pvt. Ltd.
Document ID: HR-POL-008
Version: 1.0
Effective Date: January 1, 2026
Title: Leave Policy
Classification: Confidential – Internal Use Only

1. Objective
This policy defines the types, eligibility, accrual, utilization, and administration of leave for employees of ABC Solutions Pvt. Ltd. in compliance with the applicable State Shops and Establishments Acts, the Maternity Benefit Act, 1961, and other relevant labour laws. The objective is to ensure u


In [5]:
#Improved Chunking (With Overlap)

In [6]:
def chunk_with_overlap(text, chunk_size=500, overlap=100):
    chunks = []
    step = chunk_size - overlap
    
    for i in range(0, len(text), step):
        chunks.append(text[i:i+chunk_size])
    
    return chunks

In [7]:
'''
Overlap ensures context continuity across chunks, which significantly improves retrieval quality
'''

chunks = chunk_with_overlap(sample_text)

print("Total chunks:", len(chunks))
print(chunks[0])
print("----")
print(chunks[1])

Total chunks: 9

ABC Solutions Pvt. Ltd.
ABC Solutions Pvt. Ltd.
Document ID: HR-POL-008
Version: 1.0
Effective Date: January 1, 2026
Title: Leave Policy
Classification: Confidential – Internal Use Only

1. Objective
This policy defines the types, eligibility, accrual, utilization, and administration of leave for employees of ABC Solutions Pvt. Ltd. in compliance with the applicable State Shops and Establishments Acts, the Maternity Benefit Act, 1961, and other relevant labour laws. The objective is to ensure u
----
 Acts, the Maternity Benefit Act, 1961, and other relevant labour laws. The objective is to ensure uniform application of leave benefits, workforce continuity, and statutory compliance, while providing adequate rest and recovery time to employees.

2. Scope
This policy applies to all confirmed employees across all grades and locations of ABC Solutions Pvt. Ltd. Leave during probation shall be governed by the terms of appointment and the Employee Onboarding Policy (HR-POL-00

In [8]:
#Enterprise-Level Chunking (Simple Version)

In [9]:
def section_based_chunking(text):
    sections = text.split("\n\n")  # split by paragraphs
    
    chunks = []
    for sec in sections:
        if len(sec.strip()) > 100:
            chunks.append(sec.strip())
    
    return chunks

In [10]:

'''
In enterprise systems, 
we chunk based on semantic boundaries like sections, clauses, or policies—not arbitrary character limits.
'''
chunks = section_based_chunking(sample_text)

print("Total chunks:", len(chunks))
print(chunks[0])

Total chunks: 9
ABC Solutions Pvt. Ltd.
ABC Solutions Pvt. Ltd.
Document ID: HR-POL-008
Version: 1.0
Effective Date: January 1, 2026
Title: Leave Policy
Classification: Confidential – Internal Use Only


In [11]:
#Adding Metadata...semantic layer + filtering

In [12]:
chunked_data = []

for file, text in documents.items():
    chunks = chunk_with_overlap(text)
    
    for i, chunk in enumerate(chunks):
        chunked_data.append({
            "text": chunk,
            "source": file,
            "chunk_id": i,
            "department": "HR" if "hr" in file.lower() else "General"
        })

len(chunked_data)

20

In [13]:
'''
We transformed raw documents into structured, context-preserving, metadata-enriched chunks 
ready for embedding and retrieval

Raw Docs → Text → Smart Chunks → Metadata → (Next: Embeddings)
Chunks → Vectors → Searchable Intelligence

'''

'\nWe transformed raw documents into structured, context-preserving, metadata-enriched chunks \nready for embedding and retrieval\n\nRaw Docs → Text → Smart Chunks → Metadata → (Next: Embeddings)\nChunks → Vectors → Searchable Intelligence\n\n'

In [14]:
import boto3
import json

bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1"
)

In [15]:
bedrock_control = boto3.client("bedrock")

models = bedrock_control.list_foundation_models()

for m in models["modelSummaries"]:
    print(m["modelId"])

nvidia.nemotron-nano-12b-v2
qwen.qwen3-coder-next
anthropic.claude-sonnet-4-20250514-v1:0
anthropic.claude-haiku-4-5-20251001-v1:0
moonshotai.kimi-k2.5
openai.gpt-oss-120b-1:0
stability.stable-creative-upscale-v1:0
qwen.qwen3-next-80b-a3b
deepseek.v3.2
amazon.nova-2-multimodal-embeddings-v1:0
nvidia.nemotron-nano-3-30b
anthropic.claude-sonnet-4-6
minimax.minimax-m2
zai.glm-4.7-flash
mistral.voxtral-mini-3b-2507
amazon.nova-pro-v1:0
stability.stable-image-remove-background-v1:0
stability.stable-image-control-sketch-v1:0
amazon.nova-2-lite-v1:0
amazon.nova-2-lite-v1:0:256k
stability.stable-conservative-upscale-v1:0
minimax.minimax-m2.5
google.gemma-3-12b-it
stability.stable-image-search-recolor-v1:0
moonshot.kimi-k2-thinking
mistral.mistral-large-3-675b-instruct
twelvelabs.pegasus-1-2-v1:0
amazon.nova-2-sonic-v1:0
mistral.devstral-2-123b
minimax.minimax-m2.1
nvidia.nemotron-super-3-120b
qwen.qwen3-32b-v1:0
mistral.ministral-3-14b-instruct
anthropic.claude-opus-4-6-v1
writer.palmyra-x5-v1

In [16]:
def get_embedding(text):
    response = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v1",
        body=json.dumps({
            "inputText": text
        })
    )
    
    result = json.loads(response['body'].read())
    return result['embedding']

In [17]:
sample_chunk = chunked_data[0]["text"]

embedding = get_embedding(sample_chunk)

print("Embedding length:", len(embedding))
print(embedding[:10])  # preview

Embedding length: 1536
[0.58203125, 0.18359375, -0.25390625, -0.37890625, -0.072265625, 0.30078125, 0.08154296875, 0.000347137451171875, 0.11181640625, 0.08935546875]


In [18]:
for item in chunked_data[:20]:
    item["embedding"] = get_embedding(item["text"])

In [19]:
len([x for x in chunked_data if "embedding" in x])

20

In [20]:
#Similarity Function

In [21]:
import numpy as np

def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

In [22]:
#Retrieval Function

In [23]:
'''
Instead of searching keywords, we are retrieving based on meaning. 
Even if wording changes, relevant content is still found.
'''


def retrieve_top_k(query, k=3):
    query_embedding = get_embedding(query)
    
    scores = []
    
    for item in chunked_data:
        if "embedding" not in item:
            continue
        
        score = cosine_similarity(query_embedding, item["embedding"])
        
        scores.append((score, item))
    
    # Sort by highest similarity
    scores.sort(key=lambda x: x[0], reverse=True)
    
    return scores[:k]

In [24]:
results = retrieve_top_k("What is leave policy?")

for score, item in results:
    print("\nScore:", score)
    print("Source:", item["source"])
    print("Text:", item["text"][:300])


Score: 0.6778227627111753
Source: HR-POL-008_Leave_Policy.docx
Text: rned by the terms of appointment and the Employee Onboarding Policy (HR-POL-001). The policy is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO).

3. Definitions
3.1 Earned Leave (EL): Leave accrued based on days worked.
3.2 Casual Leave 

Score: 0.6249072217715607
Source: HR-POL-008_Leave_Policy.docx
Text: 
ABC Solutions Pvt. Ltd.
ABC Solutions Pvt. Ltd.
Document ID: HR-POL-008
Version: 1.0
Effective Date: January 1, 2026
Title: Leave Policy
Classification: Confidential – Internal Use Only

1. Objective
This policy defines the types, eligibility, accrual, utilization, and administration of leave for e

Score: 0.6248352959428981
Source: HR-POL-008_Leave_Policy.docx
Text: Absence and Misuse
7.1 Absence without approved leave for more than three consecutive working days shall be treated as unauthorized and may attract disciplinary action.
7.2 Misrepresentati

In [25]:
#Metadata Filtering

In [26]:
def retrieve_with_filter(query, department=None, k=3):
    query_embedding = get_embedding(query)
    
    scores = []
    
    for item in chunked_data:
        if "embedding" not in item:
            continue
        
        if department and item["department"] != department:
            continue
        
        score = cosine_similarity(query_embedding, item["embedding"])
        scores.append((score, item))
    
    scores.sort(key=lambda x: x[0], reverse=True)
    
    return scores[:k]

In [27]:
'''
This is the semantic layer—combining vector search with business filters like department or document type
'''


results = retrieve_with_filter("leave policy", department="HR")

for score, item in results:
    print("\n", score, item["source"])


 0.6521293529722348 HR-POL-008_Leave_Policy.docx

 0.6378384879034483 HR-POL-008_Leave_Policy.docx

 0.6123830320846446 HR-POL-008_Leave_Policy.docx


In [28]:
'''
Achieved output
Query → Embedding → Similarity Search → Top Relevant Chunks
S3 → Extraction → Chunking → Embeddings → Retrieval

'''

'\nAchieved output\nQuery → Embedding → Similarity Search → Top Relevant Chunks\nS3 → Extraction → Chunking → Embeddings → Retrieval\n\n'

In [29]:
# RAG (LLM + Context Injection)
'''
Retrieved chunks → LLM → Final Answer

'''

'\nRetrieved chunks → LLM → Final Answer\n\n'

In [30]:
def build_context(results):
    context = ""
    
    for r in results:
        context += f"\n[Source: {r['source']}]\n"
        context += r["text"] + "\n"
    
    return context

In [31]:
def generate_response(query, context):
    response = bedrock.invoke_model(
        modelId="amazon.nova-lite-v1:0",
        body=json.dumps({
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "text": f"""
You are an enterprise AI assistant.

Answer ONLY from the provided context.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{query}
"""
                        }
                    ]
                }
            ],
            "inferenceConfig": {
                "maxTokens": 300,
                "temperature": 0.2
            }
        })
    )
    
    result = json.loads(response['body'].read())
    return result['output']['message']['content'][0]['text']

In [32]:
#reranking function

In [33]:
def rerank_results_llm(query, results, top_k=3):
    
    # results = [(score, item), ...]
    
    # Step 1: Prepare documents
    docs = ""
    for i, (score, item) in enumerate(results):
        docs += f"\nDocument {i}:\n{item['text']}\n"

    # Step 2: Prompt
    prompt = f"""
You are an expert reranking system.

Given a query and multiple documents, rank the top {top_k} most relevant documents.

Return ONLY the document numbers in order of relevance.

Query:
{query}

Documents:
{docs}

Answer format:
[2, 0, 1]
"""

    # Step 3: Call Nova (no approval needed)
    response = bedrock.invoke_model(
        modelId="amazon.nova-lite-v1:0",
        body=json.dumps({
            "messages": [
                {
                    "role": "user",
                    "content": [{"text": prompt}]
                }
            ],
            "inferenceConfig": {
                "maxTokens": 100,
                "temperature": 0
            }
        })
    )

    result = json.loads(response['body'].read())
    output = result['output']['message']['content'][0]['text']

    # Step 4: Extract indices safely
    import re
    indices = list(map(int, re.findall(r'\d+', output)))

    # Step 5: Map back to original results
    reranked = []
    for idx in indices[:top_k]:
        if idx < len(results):
            score, item = results[idx]
            reranked.append({
                "text": item["text"],
                "source": item["source"],
                "score": score
            })

    return reranked

In [34]:
def rag_pipeline(query):
    # Step 1: Retrieve
    results = retrieve_top_k(query, k=3)

    # Step 2: Rerank
    final_results = rerank_results_llm(query, results, top_k=3)
    
    # Step 3: Build context
    context = build_context(final_results)
    
    # Step 4: Generate answer
    answer = generate_response(query, context)
    
    return answer, results

In [35]:
query = "What is the leave policy?"

answer, results = rag_pipeline(query)

print("ANSWER:\n", answer)

ANSWER:
 The leave policy includes various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), Maternity Leave, and Paternity Leave. It also outlines rules for absence, misuse of leave, and cross-policy dependencies. For example, absence without approved leave for more than three consecutive working days is treated as unauthorized and may attract disciplinary action. Additionally, the policy is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO).


In [36]:
bedrock_control = boto3.client("bedrock")

models = bedrock_control.list_foundation_models()

for m in models["modelSummaries"]:
    print(m["modelId"])

nvidia.nemotron-nano-12b-v2
qwen.qwen3-coder-next
anthropic.claude-sonnet-4-20250514-v1:0
anthropic.claude-haiku-4-5-20251001-v1:0
moonshotai.kimi-k2.5
openai.gpt-oss-120b-1:0
stability.stable-creative-upscale-v1:0
qwen.qwen3-next-80b-a3b
deepseek.v3.2
amazon.nova-2-multimodal-embeddings-v1:0
nvidia.nemotron-nano-3-30b
anthropic.claude-sonnet-4-6
minimax.minimax-m2
zai.glm-4.7-flash
mistral.voxtral-mini-3b-2507
amazon.nova-pro-v1:0
stability.stable-image-remove-background-v1:0
stability.stable-image-control-sketch-v1:0
amazon.nova-2-lite-v1:0
amazon.nova-2-lite-v1:0:256k
stability.stable-conservative-upscale-v1:0
minimax.minimax-m2.5
google.gemma-3-12b-it
stability.stable-image-search-recolor-v1:0
moonshot.kimi-k2-thinking
mistral.mistral-large-3-675b-instruct
twelvelabs.pegasus-1-2-v1:0
amazon.nova-2-sonic-v1:0
mistral.devstral-2-123b
minimax.minimax-m2.1
nvidia.nemotron-super-3-120b
qwen.qwen3-32b-v1:0
mistral.ministral-3-14b-instruct
anthropic.claude-opus-4-6-v1
writer.palmyra-x5-v1

In [37]:
'''
Enterprise-grade RAG

'''

'\nEnterprise-grade RAG\n\n'

In [38]:
query = "What is Eligibility for remote work?"

answer, results = rag_pipeline(query)

print("ANSWER:\n", answer)

ANSWER:
 Eligibility for remote work includes:

1. Employees must have a minimum of 6 months of confirmed service.
2. Employees must have a most recent Overall Performance Rating (OPR) of 3 or above as per the Performance Appraisal Policy (HR-POL-006).
3. Employees must not have any active disciplinary actions or be under a Performance Improvement Plan (PIP).


# The LLM is not the intelligence—the pipeline design is. Retrieval, reranking, and prompt control determine system quality

In [ ]:
## STEP 2 - Wrap HR RAG as a Strands Tool

In [42]:
test_answer, test_results = rag_pipeline("What is the leave policy?")
print(test_answer)

The leave policy includes various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), Maternity Leave, and Paternity Leave. It also outlines rules for absence without approved leave, misrepresentation of medical certificates, and the impact of excessive unauthorized absence on performance appraisal scores. Additionally, it mentions cross-policy dependencies such as linkage with insurance claims for long-term medical leave and the consideration of temporary work-from-home during medical recovery. The policy is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO).


In [44]:
def hr_rag_tool(query: str) -> str:
    """
    Wrapper around the existing HR RAG chatbot.
    Returns only the final answer text for the agent.
    """
    result = rag_pipeline(query)

    # Handle tuple-style output: (answer, results)
    if isinstance(result, tuple):
        return str(result[0])

    return str(result)

In [54]:
print(hr_rag_tool("What is Eligibility for remote work?"))

Eligibility for remote work includes:

1. Minimum of 6 months of confirmed service.
2. Most recent Overall Performance Rating (OPR) of 3 or above as per Performance Appraisal Policy (HR-POL-006).
3. No active disciplinary actions or Performance Improvement Plans (PIP).

Employees on probation, under PIP, or under disciplinary review are excluded unless explicitly approved by the Chief Human Resources Officer (CHRO).


In [48]:
from strands import Agent
from strands import tool

In [55]:
@tool
def hr_rag_search(query: str) -> str:
    """
    Use the enterprise HR knowledge base to answer policy-related HR questions.
    Use this tool for questions about leave policy, contractor rules, HR documents,
    and other HR knowledge queries.
    """
    result = rag_pipeline(query)

    if isinstance(result, tuple):
        return str(result[0])

    return str(result)

In [56]:
agent = Agent(
    model="amazon.nova-lite-v1:0",
    tools=[hr_rag_search],
    system_prompt="""
You are an enterprise HR assistant.

RULES:
1. For any HR policy or HR document question, always call hr_rag_search.
2. Never answer HR policy questions from your own knowledge.
3. Use the tool for leave policy, contractor policy, benefits, attendance, and HR process questions.
4. Keep answers concise and grounded in the retrieved enterprise content.
"""
)

In [58]:
response = agent("What is Eligibility for remote work?")
print(str(response.message))

Eligibility for remote work includes:

1. A minimum of 6 months of confirmed service.
2. A most recent Overall Performance Rating (OPR) of 3 or above as per Performance Appraisal Policy (HR-POL-006).
3. No active disciplinary actions or Performance Improvement Plans (PIP).{'role': 'assistant', 'content': [{'text': 'Eligibility for remote work includes:\n\n1. A minimum of 6 months of confirmed service.\n2. A most recent Overall Performance Rating (OPR) of 3 or above as per Performance Appraisal Policy (HR-POL-006).\n3. No active disciplinary actions or Performance Improvement Plans (PIP).'}]}


In [62]:
#Step 3: add the leave-apply tool

leave_requests = []

In [63]:
#Leave apply function

def apply_leave_logic(employee_id: str, days: int, reason: str = "Not specified") -> str:
    """
    Simple demo leave application logic.
    Stores leave requests in memory.
    """
    record = {
        "employee_id": employee_id,
        "days": days,
        "reason": reason
    }
    
    leave_requests.append(record)
    
    return f"Leave applied successfully for {days} day(s) for employee {employee_id}. Reason: {reason}"

In [71]:
print(apply_leave_logic("EMP001", 2, "Personal work"))
print(leave_requests)

Leave applied successfully for 2 day(s) for employee EMP001. Reason: Personal work
[{'employee_id': 'EMP001', 'days': 2, 'reason': 'Personal work'}, {'employee_id': 'EMP001', 'days': 2, 'reason': 'personal work'}, {'employee_id': 'EMP001', 'days': 2, 'reason': 'Personal work'}]


In [72]:
@tool
def apply_leave(days: int, reason: str = "Not specified", employee_id: str = "EMP001") -> str:
    """
    Apply leave for the current employee.

    Use this tool when the user wants to request or apply leave.
    The 'days' parameter is the number of leave days requested.
    The 'reason' is the leave reason.
    If employee_id is not provided, use EMP001.
    """
    return apply_leave_logic(employee_id, days, reason)

In [73]:
agent = Agent(
    model="amazon.nova-lite-v1:0",
    tools=[hr_rag_search, apply_leave],
    system_prompt="""
You are an enterprise HR assistant.

Rules:
1. Use hr_rag_search for HR policy, HR documentation, leave rules, contractor rules, and other HR knowledge questions.
2. Use apply_leave when the user wants to request leave or apply leave.
3. Do not expose your internal reasoning.
4. Do not mention tool names unless explicitly asked.
5. Return only the final answer for the user.
"""
)

In [74]:
response = agent("What is the leave policy?")
print(response.message)

<thinking>The user is asking for the leave policy, which falls under HR knowledge queries. I need to use the hr_rag_search tool to provide the relevant information.</thinking>

Tool #1: hr_rag_search
The leave policy includes various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), Maternity Leave, and Paternity Leave. It also outlines rules for absence without approved leave, misrepresentation of medical certificates, and the impact of excessive unauthorized absence on performance appraisal scores. Additionally, it mentions cross-policy dependencies with Health Benefits and Remote Work policies. The policy is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO).{'role': 'assistant', 'content': [{'text': 'The leave policy includes various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), Maternity Leave, and Paternity Leave. It also outlines rules for absence without appro

In [75]:
response = agent("Apply leave for 2 days for employee EMP001 because of personal work.")
print(response.message)

<thinking>The user wants to apply for leave. I need to use the apply_leave tool with the specified parameters.</thinking> 
Tool #2: apply_leave
Your leave for 2 days has been successfully applied for employee EMP001 due to personal work.{'role': 'assistant', 'content': [{'text': 'Your leave for 2 days has been successfully applied for employee EMP001 due to personal work.'}]}


In [76]:
print(leave_requests)

[{'employee_id': 'EMP001', 'days': 2, 'reason': 'Personal work'}, {'employee_id': 'EMP001', 'days': 2, 'reason': 'personal work'}, {'employee_id': 'EMP001', 'days': 2, 'reason': 'Personal work'}, {'employee_id': 'EMP001', 'days': 2, 'reason': 'personal work'}]


In [77]:
## STEP 4 - Multi-step orchestration and memory/state

In [78]:
def get_leave_summary_logic(employee_id: str = "EMP001") -> str:
    """
    Summarize leave applied by the employee from in-memory state.
    """
    employee_records = [r for r in leave_requests if r["employee_id"] == employee_id]

    if not employee_records:
        return f"No leave applications found for employee {employee_id}."

    total_days = sum(r["days"] for r in employee_records)

    lines = [f"Employee {employee_id} has applied for a total of {total_days} day(s) of leave."]
    lines.append("Leave history:")

    for i, record in enumerate(employee_records, start=1):
        lines.append(
            f"{i}. {record['days']} day(s) - Reason: {record['reason']}"
        )

    return "\n".join(lines)

In [79]:
print(get_leave_summary_logic("EMP001"))

Employee EMP001 has applied for a total of 8 day(s) of leave.
Leave history:
1. 2 day(s) - Reason: Personal work
2. 2 day(s) - Reason: personal work
3. 2 day(s) - Reason: Personal work
4. 2 day(s) - Reason: personal work


In [80]:
@tool
def get_leave_summary(employee_id: str = "EMP001") -> str:
    """
    Get the leave application summary and total leave days for an employee.

    Use this tool when the user asks:
    - how many days of leave they applied for
    - what their leave history is
    - to show previously applied leave
    """
    return get_leave_summary_logic(employee_id)

In [81]:
agent = Agent(
    model="amazon.nova-lite-v1:0",
    tools=[hr_rag_search, apply_leave, get_leave_summary],
    system_prompt="""
You are an enterprise HR assistant.

Rules:
1. Use hr_rag_search for HR policy, HR documents, leave rules, remote work rules, and other HR knowledge questions.
2. Use apply_leave when the user wants to request or apply leave.
3. Use get_leave_summary when the user asks how many leave days they applied for, asks for leave history, or asks about previously applied leave.
4. If a user request contains multiple intents, complete all required steps in sequence.
5. Do not expose internal reasoning.
6. Do not mention tool names unless explicitly asked.
7. Return only the final answer for the user.
"""
)

In [82]:
response = agent("How many days did I apply?")
print(response.message)

<thinking>I need to retrieve the leave application summary for the user to determine how many days of leave they have applied for.</thinking>

Tool #1: get_leave_summary
You have applied for a total of 8 days of leave. Here is the leave history:
1. 2 days - Reason: Personal work
2. 2 days - Reason: personal work
3. 2 days - Reason: Personal work
4. 2 days - Reason: personal work{'role': 'assistant', 'content': [{'text': 'You have applied for a total of 8 days of leave. Here is the leave history:\n1. 2 days - Reason: Personal work\n2. 2 days - Reason: personal work\n3. 2 days - Reason: Personal work\n4. 2 days - Reason: personal work'}]}


In [83]:
response = agent("Show my leave history.")
print(response.message)

<thinking>I already have the leave history for the user from the previous query, so I will provide that information directly.</thinking>

Your leave history is as follows:
1. 2 days - Reason: Personal work
2. 2 days - Reason: personal work
3. 2 days - Reason: Personal work
4. 2 days - Reason: personal work{'role': 'assistant', 'content': [{'text': '<thinking>I already have the leave history for the user from the previous query, so I will provide that information directly.</thinking>\n\nYour leave history is as follows:\n1. 2 days - Reason: Personal work\n2. 2 days - Reason: personal work\n3. 2 days - Reason: Personal work\n4. 2 days - Reason: personal work'}]}


In [84]:
response = agent("Check the leave policy and apply leave for 2 days due to personal work.")
print(response.message)

<thinking>First, I need to search the leave policy to understand the rules and requirements. Then, I will apply for the leave based on the provided information.</thinking>

Tool #2: hr_rag_search
<thinking>I now have the leave policy information and will proceed to apply for the leave as requested.</thinking> 
Tool #3: apply_leave
Your leave for 2 days due to personal work has been applied successfully.{'role': 'assistant', 'content': [{'text': 'Your leave for 2 days due to personal work has been applied successfully.'}]}


In [85]:
response = agent("How many days did I apply?")
print(response.message)

<thinking>I need to retrieve the leave application summary for the user to determine how many days of leave they have applied for, including the new leave request.</thinking> 
Tool #4: get_leave_summary
You have applied for a total of 10 days of leave. Here is the updated leave history:
1. 2 days - Reason: Personal work
2. 2 days - Reason: personal work
3. 2 days - Reason: Personal work
4. 2 days - Reason: personal work
5. 2 days - Reason: personal work{'role': 'assistant', 'content': [{'text': 'You have applied for a total of 10 days of leave. Here is the updated leave history:\n1. 2 days - Reason: Personal work\n2. 2 days - Reason: personal work\n3. 2 days - Reason: Personal work\n4. 2 days - Reason: personal work\n5. 2 days - Reason: personal work'}]}


In [86]:
from mcp.server.fastmcp import FastMCP

In [ ]:
##  Creating real MCP server for leave tools

In [87]:
!pip install -U mcp fastapi uvicorn

  Attempting uninstall: uvicorn
    Found existing installation: uvicorn 0.42.0
    Uninstalling uvicorn-0.42.0:
      Successfully uninstalled uvicorn-0.42.0
  Attempting uninstall: fastapi
    Found existing installation: fastapi 0.135.2
    Uninstalling fastapi-0.135.2:
      Successfully uninstalled fastapi-0.135.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [fastapi]m1/2 [fastapi]


In [89]:
'''
#MCP server runs as a separate process, it should not depend on your notebook’s in-memory leave_requests list.
Storing in small json file
'''


import json
import os

LEAVE_DB_PATH = "/home/ec2-user/SageMaker/leave_requests.json"

if not os.path.exists(LEAVE_DB_PATH):
    with open(LEAVE_DB_PATH, "w") as f:
        json.dump([], f)

print("Leave DB ready:", LEAVE_DB_PATH)

Leave DB ready: /home/ec2-user/SageMaker/leave_requests.json


In [ ]:
# MCP server script

In [90]:
mcp_server_code = r'''
import json
import os
from mcp.server.fastmcp import FastMCP

LEAVE_DB_PATH = "/home/ec2-user/SageMaker/leave_requests.json"

mcp = FastMCP("HR Leave MCP Server")


def load_requests():
    if not os.path.exists(LEAVE_DB_PATH):
        return []
    with open(LEAVE_DB_PATH, "r") as f:
        return json.load(f)


def save_requests(data):
    with open(LEAVE_DB_PATH, "w") as f:
        json.dump(data, f, indent=2)


@mcp.tool(description="Apply leave for an employee")
def apply_leave(employee_id: str = "EMP001", days: int = 1, reason: str = "Not specified") -> str:
    requests = load_requests()

    record = {
        "employee_id": employee_id,
        "days": days,
        "reason": reason
    }
    requests.append(record)
    save_requests(requests)

    return f"Leave applied successfully for {days} day(s) for employee {employee_id}. Reason: {reason}"


@mcp.tool(description="Get leave summary and leave history for an employee")
def get_leave_summary(employee_id: str = "EMP001") -> str:
    requests = load_requests()
    employee_records = [r for r in requests if r["employee_id"] == employee_id]

    if not employee_records:
        return f"No leave applications found for employee {employee_id}."

    total_days = sum(r["days"] for r in employee_records)

    lines = [f"Employee {employee_id} has applied for a total of {total_days} day(s) of leave."]
    lines.append("Leave history:")

    for i, record in enumerate(employee_records, start=1):
        lines.append(f"{i}. {record['days']} day(s) - Reason: {record['reason']}")

    return "\\n".join(lines)


if __name__ == "__main__":
    mcp.run(transport="streamable-http")
'''

In [92]:
server_path = "/home/ec2-user/SageMaker/hr_leave_mcp_server.py"

with open(server_path, "w") as f:
    f.write(mcp_server_code)

print("MCP server script written to:", server_path)

MCP server script written to: /home/ec2-user/SageMaker/hr_leave_mcp_server.py


In [ ]:
#Start the MCP server

In [94]:
import subprocess
import os
import signal
import time

SERVER_SCRIPT = "/home/ec2-user/SageMaker/hr_leave_mcp_server.py"
SERVER_LOG = "/home/ec2-user/SageMaker/hr_leave_mcp_server.log"
SERVER_PID_FILE = "/home/ec2-user/SageMaker/hr_leave_mcp_server.pid"

# Stop older server if one exists
if os.path.exists(SERVER_PID_FILE):
    with open(SERVER_PID_FILE, "r") as f:
        old_pid = int(f.read().strip())
    try:
        os.kill(old_pid, signal.SIGTERM)
        print(f"Stopped old MCP server process: {old_pid}")
    except ProcessLookupError:
        print(f"Old PID {old_pid} not running")
    except Exception as e:
        print(f"Could not stop old process {old_pid}: {e}")

# Start new server
log_file = open(SERVER_LOG, "w")

proc = subprocess.Popen(
    ["python", SERVER_SCRIPT],
    stdout=log_file,
    stderr=log_file,
    start_new_session=True
)

with open(SERVER_PID_FILE, "w") as f:
    f.write(str(proc.pid))

time.sleep(3)

print(f"MCP server started with PID: {proc.pid}")
print(f"Log file: {SERVER_LOG}")

MCP server started with PID: 23001
Log file: /home/ec2-user/SageMaker/hr_leave_mcp_server.log


In [95]:
import os
import signal

with open("/home/ec2-user/SageMaker/hr_leave_mcp_server.pid", "r") as f:
    pid = int(f.read().strip())

try:
    os.kill(pid, 0)
    print(f"Process {pid} is running")
except OSError:
    print(f"Process {pid} is NOT running")

Process 23001 is running


In [96]:
!tail -n 50 /home/ec2-user/SageMaker/hr_leave_mcp_server.log

INFO:     Started server process [23001]
INFO:     Waiting for application startup.
[04/09/26 11:57:48] INFO     StreamableHTTP       ]8;id=955703;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=30744;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\
                             session manager                                    
                             started                                            
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


In [97]:
!curl -i http://127.0.0.1:8000/mcp/ || true

HTTP/1.1 307 Temporary Redirect
date: Thu, 09 Apr 2026 12:00:20 GMT
server: uvicorn
content-length: 0
location: ]8;;http://127.0.0.1:8000/mcp\http://127.0.0.1:8000/mcp
]8;;\


In [ ]:
##  Connecting Strands agent to MCP server

In [98]:
from strands import Agent
from strands.tools.mcp import MCPClient

In [99]:
from mcp.client.streamable_http import streamablehttp_client

MCP_SERVER_URL = "http://127.0.0.1:8000/mcp/"

mcp_client = MCPClient(lambda: streamablehttp_client(MCP_SERVER_URL))
print("MCP client created")

MCP client created


In [101]:
#Fetching tools from the MCP server
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

MCP_SERVER_URL = "http://127.0.0.1:8000/mcp/"

mcp_client = MCPClient(lambda: streamablehttp_client(MCP_SERVER_URL))

agent = Agent(
    model="amazon.nova-lite-v1:0",
    tools=[hr_rag_search, mcp_client],
    system_prompt="""
You are an enterprise HR assistant.

Rules:
1. Use hr_rag_search for HR knowledge, HR policy, leave rules, remote work rules, contractor policy, and HR document questions.
2. Use MCP-provided leave tools when the user wants to apply leave or asks about leave history or total applied leave.
3. If a request contains both information and action, perform both in sequence.
4. Do not expose internal reasoning.
5. Return only the final answer for the user.
"""
)

In [102]:
#Testing MCP leave tool routing
response = agent("Apply leave for 2 days due to personal work.")
print(response.message)

<thinking>The user wants to apply for leave. I need to use the 'apply_leave' tool for this request.</thinking>

Tool #1: apply_leave
Your leave for 2 days due to personal work has been successfully applied.{'role': 'assistant', 'content': [{'text': 'Your leave for 2 days due to personal work has been successfully applied.'}]}


In [103]:
#Testing MCP leave summary routing

response = agent("How many days did I apply?")
print(response.message)

<thinking>The user wants to know how many days they have applied for leave. I need to use the 'get_leave_summary' tool to get this information.</thinking> 
Tool #2: get_leave_summary
You have applied for a total of 2 days of leave due to personal work.{'role': 'assistant', 'content': [{'text': 'You have applied for a total of 2 days of leave due to personal work.'}]}


In [104]:
# RAG HR Chatbot - local tool call

response = agent("What is the leave policy?")
print(response.message)

<thinking>The user wants to know about the leave policy. I need to use the 'hr_rag_search' tool to get this information.</thinking> 
Tool #3: hr_rag_search
The leave policy for ABC Solutions Pvt. Ltd. includes definitions and scope of various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), and Maternity/Paternity Leave. The policy applies to all confirmed employees across all grades and locations, ensuring compliance with relevant labour laws.{'role': 'assistant', 'content': [{'text': 'The leave policy for ABC Solutions Pvt. Ltd. includes definitions and scope of various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), and Maternity/Paternity Leave. The policy applies to all confirmed employees across all grades and locations, ensuring compliance with relevant labour laws.'}]}


In [105]:
#leave action, MCP tool
response = agent("Apply leave for 1 day because of a family event.")
print(response.message)

<thinking>The user wants to apply for leave. I need to use the 'apply_leave' tool for this request.</thinking> 
Tool #4: apply_leave
Your leave for 1 day due to a family event has been successfully applied.{'role': 'assistant', 'content': [{'text': 'Your leave for 1 day due to a family event has been successfully applied.'}]}


In [106]:
# state query, MCP tool
response = agent("Show my leave history.")
print(response.message)

<thinking>The user wants to know their leave history. I need to use the 'get_leave_summary' tool to get this information.</thinking> 
Tool #5: get_leave_summary
Your leave history shows: 2 days for personal work and 1 day for a family event. You have applied for a total of 3 days of leave.{'role': 'assistant', 'content': [{'text': 'Your leave history shows: 2 days for personal work and 1 day for a family event. You have applied for a total of 3 days of leave.'}]}


In [107]:
#Testing multi-step with MCP included

'''
local tool routing
MCP tool routing
multi-step orchestration together
'''

response = agent("Check the leave policy and apply leave for 1 day due to personal work.")
print(response.message)

<thinking>The user wants to know the leave policy and also apply for leave. I need to use the 'hr_rag_search' tool to get the leave policy information and the 'apply_leave' tool to apply for leave.</thinking> 
Tool #6: hr_rag_search

Tool #7: apply_leave
The Leave Policy document (HR-POL-008) outlines the types, eligibility, accrual, utilization, and administration of leave for employees of ABC Solutions Pvt. Ltd. It applies to all confirmed employees across all grades and locations, and is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO). Your leave for 1 day due to personal work has been successfully applied.{'role': 'assistant', 'content': [{'text': 'The Leave Policy document (HR-POL-008) outlines the types, eligibility, accrual, utilization, and administration of leave for employees of ABC Solutions Pvt. Ltd. It applies to all confirmed employees across all grades and locations, and is administered by the Human Resources

In [ ]:
# Agent → local HR RAG tool + MCP tool provider

In [ ]:
## Strands hook guardrail

In [108]:
from strands.hooks.events import BeforeInvocationEvent

In [ ]:
#hook guardrail callback

In [109]:
BLOCKED_KEYWORDS = [
    "salary",
    "compensation",
    "payroll",
    "pay band",
    "salary hike",
    "ctc",
    "confidential salary",
    "employee salary"
]

def salary_guardrail(event: BeforeInvocationEvent):
    """
    Strands hook guardrail that blocks salary/compensation related queries
    before the model or tools are invoked.
    """
    # event.messages is the invocation input message list
    combined_text = ""

    for msg in event.messages:
        content = msg.get("content", "")
        if isinstance(content, str):
            combined_text += " " + content
        elif isinstance(content, list):
            for item in content:
                if isinstance(item, dict) and "text" in item:
                    combined_text += " " + item["text"]

    combined_text = combined_text.lower()

    if any(keyword in combined_text for keyword in BLOCKED_KEYWORDS):
        raise ValueError(
            "Policy block: I cannot help with salary, compensation, payroll, or confidential pay information."
        )

In [111]:
agent = Agent(
    model="amazon.nova-lite-v1:0",
    tools=[hr_rag_search, mcp_client],
    system_prompt="""
You are an enterprise HR assistant.

Rules:
1. Use hr_rag_search for HR knowledge, HR policy, leave rules, remote work rules, contractor policy, and HR document questions.
2. Use MCP-provided leave tools when the user wants to apply leave or asks about leave history or total applied leave.
3. If a request contains both information and action, perform both in sequence.
4. Do not expose internal reasoning.
5. Return only the final answer for the user.
"""
)

In [112]:
from strands.hooks import BeforeInvocationEvent

agent.add_hook(salary_guardrail, BeforeInvocationEvent)

In [113]:
def safe_agent_call(query: str) -> str:
    try:
        response = agent(query)
        return response.message
    except Exception as e:
        return str(e)

In [114]:
print(safe_agent_call("What is the leave policy?"))


<thinking>The user is asking for information about the leave policy. I should use the hr_rag_search tool to find the relevant information.</thinking>

Tool #1: hr_rag_search
The leave policy includes various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), Maternity Leave, and Paternity Leave. It is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO). The policy also outlines rules for absence, misuse of leave, and cross-policy dependencies with other HR policies such as Performance Appraisal and Health Benefits.{'role': 'assistant', 'content': [{'text': 'The leave policy includes various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), Maternity Leave, and Paternity Leave. It is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO). The policy also outlines rules for absence, misuse of leave, and cross-policy depe

In [115]:
print(safe_agent_call("Show me employee salary details."))

Policy block: I cannot help with salary, compensation, payroll, or confidential pay information.


In [116]:
print(safe_agent_call("what is the salary hike in finance department?."))

Policy block: I cannot help with salary, compensation, payroll, or confidential pay information.


In [ ]:
# Strands Hook guard rails invoke before LLM calls ...similarity with llmguards opensource library
# Mention bedrock gaurdrails post invocation

In [ ]:
## STEP 6C - Observability and Traceability

In [117]:
result = agent("What is the leave policy?")
print(type(result))
print(dir(result))

<thinking>The user has already asked for the leave policy, and I have provided the information. I will repeat the information I previously provided.</thinking>

The leave policy includes various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), Maternity Leave, and Paternity Leave. It is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO). The policy also outlines rules for absence, misuse of leave, and cross-policy dependencies with other HR policies such as Performance Appraisal and Health Benefits.<class 'strands.agent.agent_result.AgentResult'>
['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__',

In [ ]:
# safe trace-aware runner

In [118]:
def run_agent_with_trace(query: str):
    try:
        result = agent(query)
        return {
            "ok": True,
            "query": query,
            "message": result.message,
            "traces": getattr(result, "traces", None),
            "raw_result": result
        }
    except Exception as e:
        return {
            "ok": False,
            "query": query,
            "message": str(e),
            "traces": None,
            "raw_result": None
        }

In [ ]:
#trace printer

In [119]:
def print_trace_node(node, indent=0):
    prefix = "  " * indent

    # Try common attributes safely
    name = getattr(node, "name", None) or getattr(node, "span_name", None) or type(node).__name__
    start_time = getattr(node, "start_time", None)
    end_time = getattr(node, "end_time", None)
    duration_ms = None

    if start_time is not None and end_time is not None:
        try:
            duration_ms = (end_time - start_time) * 1000
        except Exception:
            duration_ms = None

    if duration_ms is not None:
        print(f"{prefix}- {name} ({duration_ms:.2f} ms)")
    else:
        print(f"{prefix}- {name}")

    children = getattr(node, "children", None) or []
    for child in children:
        print_trace_node(child, indent + 1)

In [ ]:
#wrapper

In [120]:
def print_agent_traces(trace_result):
    traces = trace_result.get("traces")

    if not traces:
        print("No traces available.")
        return

    print(f"Trace count: {len(traces)}")
    for i, trace in enumerate(traces, start=1):
        print(f"\nTrace #{i}")
        print_trace_node(trace)

In [127]:
agent = Agent(
    model="amazon.nova-lite-v1:0",
    tools=[hr_rag_search, mcp_client],
    callback_handler=None,
    system_prompt="""
You are an enterprise HR assistant.

Rules:
1. Use hr_rag_search for HR knowledge, HR policy, leave rules, remote work rules, contractor policy, and HR document questions.
2. Use MCP-provided leave tools when the user wants to apply leave or asks about leave history or total applied leave.
3. If a request contains both information and action, perform both in sequence.
4. Do not expose internal reasoning.
5. Return only the final answer for the user.
"""
)

In [128]:
def extract_text_from_message(message):
    if isinstance(message, str):
        return message

    if isinstance(message, dict):
        content = message.get("content", [])
        parts = []

        for item in content:
            if isinstance(item, dict) and "text" in item:
                text = item["text"]
                # remove visible thinking blocks if present
                text = text.replace("<thinking>", "").replace("</thinking>", "")
                parts.append(text.strip())

        return "\n".join([p for p in parts if p]).strip()

    return str(message)

In [129]:
def run_agent_with_trace(query: str):
    try:
        result = agent(query)
        return {
            "ok": True,
            "query": query,
            "message": extract_text_from_message(result.message),
            "traces": getattr(result, "traces", None),
            "metrics": getattr(result, "metrics", None),
            "raw_result": result
        }
    except Exception as e:
        return {
            "ok": False,
            "query": query,
            "message": str(e),
            "traces": None,
            "metrics": None,
            "raw_result": None
        }

In [130]:
result = agent("What is the leave policy?")
print("Has traces:", hasattr(result, "traces"))
print("Traces value:", getattr(result, "traces", None))
print("Has metrics:", hasattr(result, "metrics"))
print("Metrics value:", getattr(result, "metrics", None))

Has traces: False
Traces value: None
Has metrics: True
Metrics value: EventLoopMetrics(cycle_count=2, tool_metrics={'hr_rag_search': ToolMetrics(tool={'toolUseId': 'tooluse_JUX76t1SPlzP0AcxXlpWME', 'name': 'hr_rag_search', 'input': {'query': 'What is the leave policy?'}}, call_count=1, success_count=1, error_count=0, total_time=2.043192148208618)}, cycle_durations=[2.645966053009033, 0.8867251873016357], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='ea030f74-02fb-4c7a-a53f-baecfd730e5d', usage={'inputTokens': 760, 'outputTokens': 51, 'totalTokens': 811}), EventLoopCycleMetric(event_loop_cycle_id='abef136d-7f3f-415c-a529-07cb450ca66a', usage={'inputTokens': 937, 'outputTokens': 100, 'totalTokens': 1037})], usage={'inputTokens': 1697, 'outputTokens': 151, 'totalTokens': 1848})], traces=[<strands.telemetry.metrics.Trace object at 0x7fd177450350>, <strands.telemetry.metrics.Trace object at 0x7fd186fbe420>], accumulated_usage={'inputTokens': 1697, 'out

In [131]:
def print_metrics(result):
    metrics = getattr(result, "metrics", None)

    if not metrics:
        print("No metrics available.")
        return

    print("=== METRICS ===")

    accumulated_usage = getattr(metrics, "accumulated_usage", None)
    if accumulated_usage:
        print("Token usage:", accumulated_usage)

    cycle_durations = getattr(metrics, "cycle_durations", None)
    if cycle_durations:
        print("Cycle durations:", cycle_durations)
        print("Total execution time (s):", sum(cycle_durations))

    tool_metrics = getattr(metrics, "tool_metrics", None)
    if tool_metrics:
        print("Tool metrics:")
        for tool_name, tool_data in tool_metrics.items():
            print(f"  - {tool_name}: {tool_data}")

In [132]:
result = agent("Apply leave for 2 days due to personal work.")
print("FINAL ANSWER:")
print(extract_text_from_message(result.message))

print_metrics(result)

FINAL ANSWER:
Leave has been successfully applied for 2 days due to personal work.
=== METRICS ===
Token usage: {'inputTokens': 3903, 'outputTokens': 218, 'totalTokens': 4121}
Cycle durations: [2.645966053009033, 0.8867251873016357, 0.7332251071929932, 0.41364002227783203]
Total execution time (s): 4.679556369781494
Tool metrics:
  - hr_rag_search: ToolMetrics(tool={'toolUseId': 'tooluse_JUX76t1SPlzP0AcxXlpWME', 'name': 'hr_rag_search', 'input': {'query': 'What is the leave policy?'}}, call_count=1, success_count=1, error_count=0, total_time=2.043192148208618)
  - apply_leave: ToolMetrics(tool={'toolUseId': 'tooluse_nlT5QGSgaIQnfNfZeJRY4C', 'name': 'apply_leave', 'input': {'reason': 'Personal work', 'days': 2}}, call_count=1, success_count=1, error_count=0, total_time=0.013399124145507812)


In [133]:
def demo_observe(query: str):
    try:
        result = agent(query)

        print("=" * 80)
        print("QUERY:")
        print(query)

        print("\nFINAL ANSWER:")
        print(extract_text_from_message(result.message))

        print("\nOBSERVABILITY:")
        print_metrics(result)

        traces = getattr(result, "traces", None)
        print("\nRAW TRACES AVAILABLE:", bool(traces))
        print("=" * 80)

    except Exception as e:
        print("=" * 80)
        print("QUERY:")
        print(query)
        print("\nBLOCKED / ERROR:")
        print(str(e))
        print("=" * 80)

In [134]:
demo_observe("What is the leave policy?")


QUERY:
What is the leave policy?

FINAL ANSWER:
The leave policy includes various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), Maternity Leave, and Paternity Leave. It is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO). The policy also outlines rules for absence, misuse of leave, and cross-policy dependencies with other HR policies like Performance Appraisal and Health Benefits.

OBSERVABILITY:
=== METRICS ===
Token usage: {'inputTokens': 6422, 'outputTokens': 352, 'totalTokens': 6774}
Cycle durations: [2.645966053009033, 0.8867251873016357, 0.7332251071929932, 0.41364002227783203, 2.0185418128967285, 1.1532714366912842]
Total execution time (s): 7.851369619369507
Tool metrics:
  - hr_rag_search: ToolMetrics(tool={'toolUseId': 'tooluse_8BLJyaFPl4teJpJsnwz03T', 'name': 'hr_rag_search', 'input': {'query': 'What is the leave policy?'}}, call_count=2, success_count=2, error_count=0, total_time=3

In [135]:
demo_observe("Apply leave for 2 days due to personal work.")


QUERY:
Apply leave for 2 days due to personal work.

FINAL ANSWER:
Leave has been successfully applied for 2 days due to personal work.

OBSERVABILITY:
=== METRICS ===
Token usage: {'inputTokens': 9404, 'outputTokens': 422, 'totalTokens': 9826}
Cycle durations: [2.645966053009033, 0.8867251873016357, 0.7332251071929932, 0.41364002227783203, 2.0185418128967285, 1.1532714366912842, 1.198564052581787, 0.4873311519622803]
Total execution time (s): 9.537264823913574
Tool metrics:
  - hr_rag_search: ToolMetrics(tool={'toolUseId': 'tooluse_8BLJyaFPl4teJpJsnwz03T', 'name': 'hr_rag_search', 'input': {'query': 'What is the leave policy?'}}, call_count=2, success_count=2, error_count=0, total_time=3.3643784523010254)
  - apply_leave: ToolMetrics(tool={'toolUseId': 'tooluse_gg7ierX9Y9E8E6Dys2fs3R', 'name': 'apply_leave', 'input': {'reason': 'Personal work', 'days': 2}}, call_count=2, success_count=2, error_count=0, total_time=0.025376319885253906)

RAW TRACES AVAILABLE: False


In [136]:
demo_observe("Check the leave policy and apply leave for 1 day due to a family event.")


QUERY:
Check the leave policy and apply leave for 1 day due to a family event.

FINAL ANSWER:
The leave policy includes various types of leave such as Earned Leave (EL), Casual Leave (CL), Sick Leave (SL), Maternity Leave, and Paternity Leave. It is administered by the Human Resources Department under the authority of the Chief Human Resources Officer (CHRO). The policy also includes rules for absence without approved leave, misrepresentation of medical certificates, and misuse of leave privileges, which may lead to disciplinary action. Additionally, it mentions cross-policy dependencies with Performance Appraisal, Health Benefits, Remote Work, and Onboarding policies.

Leave has been successfully applied for 1 day due to a family event.

OBSERVABILITY:
=== METRICS ===
Token usage: {'inputTokens': 12823, 'outputTokens': 633, 'totalTokens': 13456}
Cycle durations: [2.645966053009033, 0.8867251873016357, 0.7332251071929932, 0.41364002227783203, 2.0185418128967285, 1.1532714366912842, 1.1

In [137]:
demo_observe("Show me employee salary details.")

QUERY:
Show me employee salary details.

FINAL ANSWER:
The user wants to see salary details. I cannot provide salary details due to privacy policies, and it's not within the scope of the available tools. I'm sorry, but I cannot provide salary details due to privacy policies. If you have any other HR-related queries, feel free to ask.

OBSERVABILITY:
=== METRICS ===
Token usage: {'inputTokens': 14797, 'outputTokens': 701, 'totalTokens': 15498}
Cycle durations: [2.645966053009033, 0.8867251873016357, 0.7332251071929932, 0.41364002227783203, 2.0185418128967285, 1.1532714366912842, 1.198564052581787, 0.4873311519622803, 2.491748809814453, 1.4143600463867188, 0.8434269428253174]
Total execution time (s): 14.286800622940063
Tool metrics:
  - hr_rag_search: ToolMetrics(tool={'toolUseId': 'tooluse_H2DRaemrExijDgnmvddPBW', 'name': 'hr_rag_search', 'input': {'query': 'What is the leave policy?'}}, call_count=3, success_count=3, error_count=0, total_time=4.605194091796875)
  - apply_leave: ToolMe